# SQL Practice - Day 12

## 20 SQL Interview Questions

Topics: CTE, JOINs, SELF JOIN, LIKE, Window Functions, Subqueries, Correlated Subqueries, CASE WHEN, GROUP BY/HAVING, Top-N.

**Level:** 2–3 year Data Analyst

Write SQL in the blank cell under each question.

## Q1 — CTE

**Task:** Find each customer's total spending using a CTE.

In [1]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,103],'amount':[100,300,500,200,400]})
df

,customer_id,amount
0,101,100
1,101,300
2,102,500
3,103,200
4,103,400


In [4]:
from pandasql import sqldf
sqldf("""
with my_cte as (
select customer_id ,sum(amount) as total_spending
from df
group by customer_Id
)
select * from my_cte



""")

,customer_id,total_spending
0,101,400
1,102,500
2,103,600


## Q2 — CTE + Subquery

**Task:** Find customers whose total spending is greater than the average customer spending.

In [5]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,103,104],'amount':[100,300,500,200,400,700]})
df

,customer_id,amount
0,101,100
1,101,300
2,102,500
3,103,200
4,103,400
5,104,700


In [12]:
sqldf("""with my_cte as (
    select customer_id , sum(amount) as total_spending
    from df 
    group by customer_id 

    )
select * from my_cte
where total_spending > (select avg(total_spending)
                            from my_cte)
    
""")

,customer_id,total_spending
0,103,600
1,104,700


## Q3 — INNER JOIN

**Task:** Return customer name and total order amount for customers who placed orders.

In [13]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,102,103,104],'name':['A','B','C','D']})
df1=pd.DataFrame({'order_id':[1,2,3,4,5],'customer_id':[101,101,102,103,103],'amount':[100,200,300,400,500]})
print(df); print(df1)

   customer_id name
0          101    A
1          102    B
2          103    C
3          104    D
   order_id  customer_id  amount
0         1          101     100
1         2          101     200
2         3          102     300
3         4          103     400
4         5          103     500


In [20]:
sqldf("""
SELECT
    c.name,
    SUM(o.amount) AS total_order_amount
FROM df c
INNER JOIN df1 o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
""")

,name,total_order_amount
0,A,300
1,B,300
2,C,900


## Q4 — LEFT JOIN

**Task:** Find customers who have never placed an order.

In [21]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,102,103,104,105],'name':['A','B','C','D','E']})
df1=pd.DataFrame({'order_id':[1,2,3],'customer_id':[101,103,103]})
print(df); print(df1)

   customer_id name
0          101    A
1          102    B
2          103    C
3          104    D
4          105    E
   order_id  customer_id
0         1          101
1         2          103
2         3          103


In [26]:
sqldf("""
select c.name as name , count(o.order_id) as total_order
from df c
left join df1 o on c.customer_id  = o.customer_id 
where o.order_id is null
group by name 




""")

,name,total_order
0,B,0
1,D,0
2,E,0


## Q5 — SELF JOIN

**Task:** Find employees whose salary is greater than their manager's salary.

In [27]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5],'name':['A','B','C','D','E'],'manager_id':[None,1,1,None,4],'salary':[80000,90000,70000,70000,75000]})
df

,emp_id,name,manager_id,salary
0,1,A,NaN,80000
1,2,B,1.0,90000
2,3,C,1.0,70000
3,4,D,NaN,70000
4,5,E,4.0,75000


In [28]:
sqldf("""
select e.name as emp_name
from df e 
join df m on e.manager_id = m.emp_id
where e.salary > m.salary


""")

,emp_name
0,B
1,E


## Q6 — SELF JOIN

**Task:** Return each employee's name and their manager's name.

In [29]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5],'name':['A','B','C','D','E'],'manager_id':[None,1,1,4,4]})
df

,emp_id,name,manager_id
0,1,A,NaN
1,2,B,1.0
2,3,C,1.0
3,4,D,4.0
4,5,E,4.0


In [32]:
sqldf("""
select e.name as emp_name , m.name as manager_name
from df e
join df m on e.manager_id = m.emp_id



""")

,emp_name,manager_name
0,B,A
1,C,A
2,D,D
3,E,D


## Q7 — LIKE

**Task:** Find employees whose name starts with 'A'.

In [33]:
import pandas as pd
df=pd.DataFrame({'name':['Amit','Rahul','Ankit','Sara','Arjun']})
df

,name
0,Amit
1,Rahul
2,Ankit
3,Sara
4,Arjun


In [36]:
sqldf("""select name
from df
where name like 'A%'
""")

,name
0,Amit
1,Ankit
2,Arjun


## Q8 — LIKE

**Task:** Find customers whose email ends with '@gmail.com'.

In [37]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,102,103,104],'email':['a@gmail.com','b@yahoo.com','c@gmail.com','d@outlook.com']})
df

,customer_id,email
0,101,a@gmail.com
1,102,b@yahoo.com
2,103,c@gmail.com
3,104,d@outlook.com


In [38]:
sqldf("""
select customer_id 
from df
where email like '%@gmail.com'

""")

,customer_id
0,101
1,103


## Q9 — LIKE

**Task:** Find product names containing the word 'phone'.

In [39]:
import pandas as pd
df=pd.DataFrame({'product_id':[1,2,3,4],'product_name':['iPhone Case','Laptop','Smartphone','Headphones']})
df

,product_id,product_name
0,1,iPhone Case
1,2,Laptop
2,3,Smartphone
3,4,Headphones


In [41]:
sqldf("""
select product_name
from df
where product_name like '%phone%'


""")

,product_name
0,iPhone Case
1,Smartphone
2,Headphones


## Q10 — Window

**Task:** Find the highest-paid employee in each department using DENSE_RANK().

In [42]:
import pandas as pd
df=pd.DataFrame({'name':['A','B','C','D','E','F'],'department':['HR','HR','IT','IT','Sales','Sales'],'salary':[60000,80000,70000,90000,90000,85000]})
df

,name,department,salary
0,A,HR,60000
1,B,HR,80000
2,C,IT,70000
3,D,IT,90000
4,E,Sales,90000
5,F,Sales,85000


In [46]:
sqldf(""" 
select name , department , salary
from (select name , department , salary ,
        dense_rank()
        over(partition by department order by salary desc) as rn
        from df) t
where rn = 1




""")

,name,department,salary
0,B,HR,80000
1,D,IT,90000
2,E,Sales,90000


## Q11 — Window

**Task:** Find the second-highest salary in each department. Return ties.

In [47]:
import pandas as pd
df=pd.DataFrame({'name':['A','B','C','D','E','F'],'department':['HR','HR','HR','IT','IT','IT'],'salary':[50000,70000,70000,60000,80000,80000]})
df

,name,department,salary
0,A,HR,50000
1,B,HR,70000
2,C,HR,70000
3,D,IT,60000
4,E,IT,80000
5,F,IT,80000


In [50]:
sqldf("""
select * 
from (select * , dense_rank()
        over(partition by department order by salary desc) as rn
        from df) t
where rn = 2


""")

,name,department,salary,rn
0,A,HR,50000,2
1,D,IT,60000,2


## Q12 — Window

**Task:** Show each customer's order amount and previous order amount using LAG().

In [51]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,101,102,102],'order_date':['2025-01-01','2025-02-01','2025-03-01','2025-01-10','2025-02-10'],'amount':[100,300,200,500,700]})
df

,customer_id,order_date,amount
0,101,2025-01-01,100
1,101,2025-02-01,300
2,101,2025-03-01,200
3,102,2025-01-10,500
4,102,2025-02-10,700


In [ ]:
# Write your SQL here

## Q13 — Window + CASE

**Task:** Show department average salary and classify each employee as Above Average or Below Average.

In [52]:
import pandas as pd
df=pd.DataFrame({'name':['A','B','C','D','E'],'department':['HR','HR','IT','IT','IT'],'salary':[50000,70000,60000,90000,65000]})
df

,name,department,salary
0,A,HR,50000
1,B,HR,70000
2,C,IT,60000
3,D,IT,90000
4,E,IT,65000


In [ ]:
sqldf("""



""")

## Q14 — Subquery

**Task:** Find employees earning more than the overall average salary.

In [53]:
import pandas as pd
df=pd.DataFrame({'name':['A','B','C','D'],'salary':[50000,70000,60000,90000]})
df

,name,salary
0,A,50000
1,B,70000
2,C,60000
3,D,90000


In [55]:
sqldf("""
select name 
from df
where salary > (select avg(salary) 
                from df)



""")

,name
0,B
1,D


## Q15 — Correlated Subquery

**Task:** Find products whose price is greater than the average price of their category.

In [56]:
import pandas as pd
df=pd.DataFrame({'product_id':[1,2,3,4,5,6],'category':['A','A','A','B','B','B'],'price':[100,200,300,250,400,500]})
df

,product_id,category,price
0,1,A,100
1,2,A,200
2,3,A,300
3,4,B,250
4,5,B,400
5,6,B,500


In [58]:
sqldf("""
select product_id 
from df a
where price > (select avg(price)
                from df b 
                where a.category = b.category)


""")

,product_id
0,3
1,5
2,6


## Q16 — CASE

**Task:** Classify orders as High (>=1000), Medium (500–999), or Low (<500).

In [59]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5],'amount':[300,500,800,1000,1500]})
df

,order_id,amount
0,1,300
1,2,500
2,3,800
3,4,1000
4,5,1500


In [64]:
sqldf("""
select order_id , amount,
case
    when amount >= 1000 then 'High'
    when amount >=500 and amount <=999 then 'Medium'
    else 'Low'
end as buket
from df




""")

,order_id,amount,buket
0,1,300,Low
1,2,500,Medium
2,3,800,Medium
3,4,1000,High
4,5,1500,High


## Q17 — CASE + Aggregation

**Task:** For each customer, count Premium orders (>=1000) and Regular orders (<1000).

In [65]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,102,103,103],'amount':[500,1500,1200,300,800,2000]})
df

,customer_id,amount
0,101,500
1,101,1500
2,102,1200
3,102,300
4,103,800
5,103,2000


## Q18 — Top-N

**Task:** Find the top 3 customers by total spending using DENSE_RANK().

In [ ]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,104,104,105],'amount':[500,500,1200,1500,700,800,1500]})
df

In [75]:
sqldf("""
select customer_id ,sum(amount) ,
        dense_rank()
        over(order by sum(amount) desc) as rn
        from df t
        group by customer_id

    


""")

,customer_id,sum(amount),rn
0,103,2800,1
1,101,2000,2
2,102,1500,3


## Q19 — CTE + Window

**Task:** Find the second-highest customer by total spending using a CTE and DENSE_RANK().

In [76]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,104,104,105],'amount':[500,500,1200,1500,700,800,1500]})
df

,customer_id,amount
0,101,500
1,101,500
2,102,1200
3,103,1500
4,104,700
5,104,800
6,105,1500


In [83]:
sqldf("""with my_cte as (
    select customer_id , total_spending ,rn
    from (select customer_id ,sum(amount) as total_spending,
            dense_rank()
            over(order by sum(amount)desc)as rn
           from df
           group by customer_id)t

)
select * from my_cte
where rn = 2




""")

,customer_id,total_spending,rn
0,102,1200,2


## Q20 — Mixed Business Case

**Task:** Find employees who earn above their department average, classify them as High Performer, and rank them by salary within their department.

In [84]:
import pandas as pd
df=pd.DataFrame({'name':['A','B','C','D','E','F'],'department':['HR','HR','IT','IT','Sales','Sales'],'salary':[50000,70000,60000,90000,90000,70000]})
df

,name,department,salary
0,A,HR,50000
1,B,HR,70000
2,C,IT,60000
3,D,IT,90000
4,E,Sales,90000
5,F,Sales,70000


In [ ]:
# Write your SQL here